In [4]:
!pip install scikit-image scikit-learn --quiet

import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle

# Path to your preprocessed frames from the preprocessing notebook
# from egoblind_preprocessing, export /kaggle/working/egoblind_processed/images 
# and upload as a dataset for this notebook, then update PREPROCESSED_PATH
PREPROCESSED_PATH = "/kaggle/input/datasets/atharvadhupkar/frames/egoblind_processed/images" 
FEATURES_OUTPUT   = "/kaggle/working/svm_features/"

os.makedirs(FEATURES_OUTPUT, exist_ok=True)
print("Imports ok")

Imports ok


In [5]:
image_files = [f for f in os.listdir(PREPROCESSED_PATH) if f.endswith(".jpg")]

print(f"Found {len(image_files)} preprocessed frames")

Found 359 preprocessed frames


In [ ]:
def extract_hog_features(img_path):
    img = np.array(Image.open(img_path).convert("L"))  # grayscale
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True
    )
    return features


features  = []
filenames = []

print(f"Extracting HOG")

for i, fname in enumerate(image_files):
    img_path = os.path.join(PREPROCESSED_PATH, fname)
    feat     = extract_hog_features(img_path)
    features.append(feat)
    filenames.append(fname)

    if i % 100 == 0:
        print(f"  {i}/{len(image_files)} done")

features = np.array(features)
print(f"HOG feature matrix shape: {features.shape}")

print(f"Running PCA")
pca = PCA(n_components=200, random_state=42)
features_pca = pca.fit_transform(features)

print(f"Normalizing PCA")
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_pca)
print(f"PCA shape: {features_pca.shape}")

np.save(f"{FEATURES_OUTPUT}/features.npy",  features_scaled)
np.save(f"{FEATURES_OUTPUT}/filenames.npy", np.array(filenames))
print(f"Saved to {FEATURES_OUTPUT}")

Extracting HOG
  0/359 done
  100/359 done
  200/359 done
  300/359 done
HOG feature matrix shape: (359, 224676)
Running PCA
